In [50]:
import re
from time import time

import spacy
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

from copy import deepcopy

In [120]:
def strip_html(textdata):
    
    soup = BeautifulSoup(textdata, "html.parser")
    return soup.get_text()


def clean_text(textdata):
    
    _only_letters_pattern = re.compile(r"[^A-Za-z0-9']+")
    _no_long_numbers_pattern = re.compile(r'\d{3,}')
    _no_multiple_quotes_pattern = re.compile(r"'+")
    _no_multiple_spaces_pattern = re.compile(r"  +")

    for i in range(len(textdata)):
    
        textdata[i] = strip_html(textdata[i])
        textdata[i] = textdata[i].lower()
        textdata[i] = _only_letters_pattern.sub(' ',textdata[i])
        textdata[i] = _no_long_numbers_pattern.sub('', textdata[i])
        textdata[i] = _no_multiple_quotes_pattern.sub("", textdata[i])
        textdata[i] = _no_multiple_spaces_pattern.sub(" ", textdata[i])
        textdata[i] = textdata[i].strip()

    return textdata

def lemmatize(text, nlp, cores:int = 4):

    stext = text[0].split()

    docs = nlp(text[0])

    tokens = [doc.text for doc in docs]
    lemmas = [doc.lemma_ for doc in docs]

    ids = []

    print("length tokens", len(tokens))
    print("length stext", len(stext))
    print()

    i = 0
    j = 0
    idx = 0
    while True:

        if tokens[i] == stext[j]:
            ids.append(idx)
            i += 1
            j += 1
        
        elif tokens[i] + tokens[i+1] == stext[j]:
            ids.append(idx)
            ids.append(idx)
            i += 2
            j += 1

        else:
            ids.append(idx)
            i += 1
            j += 1


        if i == len(tokens) or j == len(stext):
            break
    
        idx += 1
        
    print("length ids", len(ids))
    print("length tokens", len(tokens))
    print()
    print(ids)
    print(tokens)

    return tokens, lemmas, ids

In [147]:
sentence = ["I havent seen this movie dont"]
nlp = spacy.load("en_core_web_sm")

['I', 'have', 'nt', 'seen', 'this', 'movie', 'do', 'nt']
['I', 'havent', 'seen', 'this', 'movie', 'dont']

length tokens 8
length stext 6

0 0
1 1
3 2
4 3
5 4
6 5
8 6


(['I', 'have', 'nt', 'seen', 'this', 'movie', 'do', 'nt'],
 ['I', 'have', 'not', 'see', 'this', 'movie', 'do', 'not'],
 [0, 1, 1, 2, 3, 4, 5, 5])

In [155]:
def do(b : int):

    print(b)

do([1, 2, 4])

[1, 2, 4]


In [45]:
nlp = spacy.load("en_core_web_sm")
lemmatizer = nlp.get_pipe("lemmatizer")

directory = '/home/kolla/projects/imdb'
df_train = pd.read_csv(f'{directory}/imdb_train.csv')
#df_train = df_train.sample(frac=1).reset_index(drop=True)
df_test = pd.read_csv(f'{directory}/imdb_test.csv')
#df_test = df_test.sample(frac=1).reset_index(drop=True)
train_data = df_train.values.tolist()
test_data = df_test.values.tolist()

X_train = [x[0] for x in train_data]
Y_train = [x[1] for x in train_data]
X_test = [x[0] for x in test_data]
Y_test = [x[1] for x in test_data]

samples = 10000

train_x = X_train[:samples]
test_x = X_test[:samples]
train_y = Y_train[:samples]
test_y = Y_test[:samples]

In [164]:
def map_tokens(stext : list, tokens : list):

    ids = []
    topw = 0
    topt = 0
    
    while True:

        print(topw, topt)

        if topt >= len(tokens) or topw >= len(stext):
            break

        elif tokens[topt] == stext[topw]:
            ids.append(topw)
            topw += 1 
            topt += 1

        elif tokens[topt] in stext[topw]:
            top_len = len(tokens[topt])  
            ids.append(topw)
            topt += 1

            while True:

                if top_len == len(stext[topw]):
                    topw += 1
                    break
                
                elif stext[topw].find(tokens[topt], top_len) == top_len:
                    ids.append(topw)
                    top_len += len(tokens[topt])
                    topt += 1

        else:
            ids.append(-1)
            topt += 1

    return ids


tokens = ["i", "l", "o", "v", "e", " ", "c", "a", "t", "s"]
stext = ["i", "love", "cats"]

ids = map_tokens(stext, tokens)
print(ids)

0 0
1 1
2 5
2 6
3 10
[0, 1, 1, 1, 1, -1, 2, 2, 2, 2]


In [47]:
train_x_lemmas_joined = [" ".join(x) for x in train_x_lemmas]
test_x_lemmas_joined = [" ".join(x) for x in test_x_lemmas]

cv = CountVectorizer(binary=True,
                         max_features=5000,
                         min_df=5,
                         max_df=0.8,
                         stop_words='english')

train_x_bin = cv.fit_transform(train_x_lemmas_joined)
test_x_bin = cv.transform(test_x_lemmas_joined)

clf = LogisticRegression().fit(train_x_bin, train_y)
print("Training Accuracy: %s" % clf.score(train_x_bin, train_y))
print("Test Accuracy: %s" % clf.score(test_x_bin, test_y))

Training Accuracy: 0.9898
Test Accuracy: 0.8425


/home/kolla/anaconda3/envs/forstaenv/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [58]:
def allign_tokens_labels_weights(tokens, vocab_weights, sentiment, threshold):
    
    alligned_tokens = []
    alligned_weights = []

    for i in range(1, len(tokens) + 1):

        bigram = tokens[i-1] + " " + tokens[i] if i < len(tokens) else None
        unigram = tokens[i-1]

        if bigram in vocab_weights.keys():
            alligned_tokens.append(bigram)
            alligned_weights.append(vocab_weights[bigram])

        elif unigram in vocab_weights.keys():
            alligned_tokens.append(unigram)
            alligned_weights.append(vocab_weights[unigram])
        
        else:
            alligned_tokens.append(unigram)
            alligned_weights.append(0.0)

    for j in range(1, len(alligned_tokens)-1):
        
        if " " in alligned_tokens[j] and " " not in alligned_tokens[j-1] and alligned_tokens[j-1] in alligned_tokens[j]:
            alligned_tokens[j-1] = "#"
            weight = alligned_weights[j-1]
            alligned_weights[j-1] = "#"
            alligned_weights[j] += weight
            
        elif " " in alligned_tokens[j] and " " in alligned_tokens[j-1] and alligned_tokens[j-1].split(" ")[1] in alligned_tokens[j]:
            alligned_tokens[j-1] = alligned_tokens[j-1].split(" ")[0]
                
        if alligned_tokens[j] in alligned_tokens[j-1]:
            alligned_tokens[j] = "#"
            weight = alligned_weights[j-1]
            alligned_weights[j] = "#"
            alligned_weights[j-1] += weight
    

    alligned_tokens = [token for token in alligned_tokens if token != "#"]
    alligned_weights = [weight for weight in alligned_weights if weight != "#"]

    if sentiment == 1:
        alligned_labels = [2 if x > threshold else 1 if x < -threshold else 0 for x in alligned_weights]
    else:
        alligned_labels = [1 if x > threshold else 2 if x < -threshold else 0 for x in alligned_weights]
    
    
    return alligned_tokens, alligned_weights, alligned_labels

vocabulary = cv.get_feature_names_out()
weights = clf.coef_[0]
vocabulary_weights = {f"{word}" : weight for word, weight in zip(vocabulary, weights)}

# train_x_lemmas
# train_x_tokens
# train_x_lemmas_joined
# train_x_bin
# train_x_clean

x_alligned_tokens, x_alligned_weights, x_alligned_labels = allign_tokens_labels_weights(train_x_lemmas[0], vocabulary_weights, train_y[0], 0.2)


def test_allign_tokens_labels_weights():
    tokens = ["i", "have", "not", "seen", "the", "movie", "yet", "and", "i", "know", "it", "is", "10", "times", "better", "than", "this", "crap"]
    vocab_weights = {"i have": 0.1, "have not": 0.2, "not seen": 0.3, "seen the": 0.4, "the movie": 0.5, "movie yet": 0.6, "yet and": 0.7, "and i": 0.8, "know it": 0.9, "it is": 1.0, "is 10": 1.1, "10 times": 1.2, "times better": 1.3, "better than": 1.4, "than this": 1.5, "this crap": 1.6}
    sentiment = 1
    threshold = 0.2
    tokens, weights, labels = allign_tokens_labels_weights(tokens, vocab_weights, sentiment, threshold)
    print(tokens)
    print(weights)
    print(labels)

    assert tokens == ['i', 'have', 'not', 'seen', 'the', 'movie', 'yet', 'and', 'i', 'know', 'it', 'is', '10', 'times', 'better', 'than', 'this', 'crap']

test_allign_tokens_labels_weights()

['i', 'have', 'not', 'seen', 'the', 'movie', 'yet', 'and i', 'know', 'it', 'is', '10', 'times', 'better', 'than', 'this crap', 'crap']
[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.6, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 0.0]
[0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0]


AssertionError: 

In [55]:
print(x_alligned_tokens)
print(x_alligned_weights)
print(x_alligned_labels)
print(train_y[0])

['i', 'like', 'how', 'this', 'start', 'out', 'feature', 'some', 'decent', 'special', 'effect', 'especially', 'for', 'a', 'film', '50', 'year', 'old', 'there', 'be', 'some', 'pretty', 'impressive', 'scenery', 'however', 'the', 'film', 'bog', 'down', 'fairly', 'early', 'on', 'with', 'some', 'very', 'dumb', 'dialog', 'as', 'the', 'male', 'all', 'try', 'to', 'flirt', 'with', 'anne', 'francis', 'altaira', 'morbius', 'view', 'this', 'in', "the'", '90', 'after', 'long', 'absence', 'it', 'be', 'fun', 'to', 'see', 'francis', 'again', 'an', 'actress', 'who', 'have', 'do', 'mostly', 'television', 'show', 'since', 'this', 'film', 'be', 'release', 'and', 'be', 'still', 'act', 'it', 'also', 'be', 'interesting', 'to', 'see', 'a', 'young', 'look', 'leslie', 'nielsen', 'dr', 'john', 'adam', 'who', 'i', "wouldn't", 'have', 'recognize', 'have', 'it', 'not', 'be', 'for', 'this', 'voice', 'watch', 'half', 'of', 'this', 'movie', 'before', 'the', 'boredom', 'come', 'almost', 'overwhelming', 'and', 'i', 'have

In [167]:
sentence = "I wouldnt say that it is a bad movie but i dont recommend it either"

nlp = spacy.load('en_core_web_sm')
lemmatizer = nlp.get_pipe("lemmatizer")

doc = nlp(sentence)

print([token.text for token in doc])
print([token.lemma_ for token in doc])
print([token.pos_ for token in doc])
print([token.tag_ for token in doc])

['I', 'would', 'nt', 'say', 'that', 'it', 'is', 'a', 'bad', 'movie', 'but', 'i', 'do', 'nt', 'recommend', 'it', 'either']
['I', 'would', 'not', 'say', 'that', 'it', 'be', 'a', 'bad', 'movie', 'but', 'I', 'do', 'not', 'recommend', 'it', 'either']
['PRON', 'AUX', 'PART', 'VERB', 'SCONJ', 'PRON', 'AUX', 'DET', 'ADJ', 'NOUN', 'CCONJ', 'PRON', 'AUX', 'PART', 'VERB', 'PRON', 'ADV']
['PRP', 'MD', 'RB', 'VB', 'IN', 'PRP', 'VBZ', 'DT', 'JJ', 'NN', 'CC', 'PRP', 'VBP', 'RB', 'VB', 'PRP', 'RB']


In [83]:
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer
import torch


tokenizer = AutoTokenizer.from_pretrained("distilroberta-base", add_prefix_space=True, device="cpu")

#"tokens" : newtokens_x,
#                     "weights" : weights_x,
#                     "text" : " ".join(newtokens_x),
#                     "sentiment" : y,
#                     "labels" : labels


data = [{"tokens" : "hello my name is eirik and i like dogs".split(),
         "weights" : [0.1, 0.2, 0.3, 0.4, 0.5],
         "text" : "hello my name is eirik and i like dogs",
         "sentiment" : 1,
         "labels" : [1, 0, 1, 1, 1, 1, 0, 0, 0]}, 
         
         {"tokens" : "hello cat".split(),
          "weights" : [0.1, 0.2],
          "text" : "hello cat",
          "sentiment" : 1,
          "labels" : [0, 0]}]


In [89]:




tokenized_inputs = tokenize_and_align_labels(data, tokenizer, "cpu")

print(tokenized_inputs)

{'input_ids': [[0, 20760, 127, 766, 16, 364, 853, 967, 8, 939, 101, 3678, 2], [0, 20760, 4758, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'labels': array([[-100,    1,    0,    1,    1,    1, -100, -100,    1,    0,    0,
           0, -100],
       [-100,    0,    0, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100]], dtype=int8), 'targets': array([[0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0],
       [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=int8), 'sentiment': array([1, 1])}


In [47]:
import numpy as np

def custom_data_collator(batch_input):

    max_len = max([len(batch) for batch in batch_input["input_ids"]])

    for i, inst in enumerate(batch_input["input_ids"]):b
        
        inst = np.concatenate([inst, np.ones(max_len - len(inst))])
        inst = inst.astype(np.int64)
        batch_input["input_ids"][i] = inst
        
    for i, inst in enumerate(batch_input["attention_mask"]):
        
        inst = np.concatenate([inst, np.zeros(max_len - len(inst))])
        inst = inst.astype(np.int64)
        batch_input["attention_mask"][i] = inst

    batch_input["input_ids"] = torch.tensor(batch_input["input_ids"]).to(device=config.device)
    batch_input["attention_mask"] = torch.tensor(batch_input["attention_mask"]).to(device=config.device)
    
    return batch_input


batch = custom_data_collator(encoded)



{'input_ids': [tensor([    0,    20,  4758, 14964,     2,     1,     1,     1,     1]), tensor([    0, 31886,  8326,     4,     2,     1,     1,     1,     1]), tensor([    0,    38,  2254,  4441, 10964,  2480,  6353,     4,     2])], 'attention_mask': [tensor([1, 1, 1, 1, 1, 0, 0, 0, 0]), tensor([1, 1, 1, 1, 1, 0, 0, 0, 0]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1])]}


/tmp/ipykernel_87084/3387169710.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch_input["input_ids"][i] = torch.cat([torch.tensor(inst), torch.ones(max_len - len(inst), dtype=torch.int64)])
/tmp/ipykernel_87084/3387169710.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch_input["attention_mask"][i] = torch.cat([torch.tensor(inst), torch.zeros(max_len - len(inst), dtype=torch.int64)])


In [96]:
# make array with all where all numbers are the same
a = np.full(5, 10)
a

array([10, 10, 10, 10, 10])

In [49]:
print(batch["input_ids"])

[tensor([    0,    20,  4758, 14964,     2,     1,     1,     1,     1]), tensor([    0, 31886,  8326,     4,     2,     1,     1,     1,     1]), tensor([    0,    38,  2254,  4441, 10964,  2480,  6353,     4,     2])]


In [8]:
from torch.nn.utils.rnn import pad_sequence
import torch


a = torch.tensor([1, 2, 3])

b = torch.tensor([1, 2, 3, 4, 5, 6])

c = [a, b]

d = pad_sequence(c,batch_first=True, padding_value=0)

d[0]

tensor([1, 2, 3, 0, 0, 0])

In [1]:
import torch

model = torch.load("/home/kolla/data/verbosius/imdb/models/imdb_model_0")

ModuleNotFoundError: No module named 'helper_functions'